# AI4000: Robotics — Lecture 31
## Computed-Torque Control and Feedback Linearization: Simulations

This notebook reproduces the simulations shown in the lecture slides and allows the parameters to be changed.

| Part | Content |
|---|---|
| **Example 1** | Mass–damper–nonlinear spring: PID control, computed-torque control, feedback linearization, comparison |
| **Example 2** | Two-link planar arm: model, independent-joint PD, computed-torque control, gain selection, tracking, model errors, feedback linearization, pole placement, animation |

**Running on Google Colab**: *File → Upload notebook*, select this file, then *Runtime → Run all*. Only `numpy`, `scipy`, `matplotlib` and `ipywidgets` are used; all are pre-installed on Colab.

All numerical results printed below match the values quoted on the slides.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
import ipywidgets as widgets

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})
d2r = np.pi / 180
C_BLUE, C_ORANGE, C_RED, C_GREEN = '#0c2344', '#e08214', '#b2182b', '#1b7837'
print('Setup complete.')

---
# Example 1: Mass–Damper–Nonlinear Spring

$$
m\ddot x+b\dot x+k_1x+k_3x^3=u
$$

with $m=1$ kg, $b=0.5$ N s/m, $k_1=1$ N/m, $k_3=20$ N/m$^3$. The input $u$ is a force; the output is the position $x$.

The **local stiffness** of the spring, $\dfrac{d}{dx}(k_1x+k_3x^3)=k_1+3k_3x^2$, grows with the amplitude:
$1.6$ N/m at $x=0.1$ m and $61$ N/m at $x=1$ m.

In [ ]:
# Plant parameters (true values)
m, b, k1, k3 = 1.0, 0.5, 1.0, 20.0

def simulate_msd(controller, xd, tf=4.0, N=801, params=(m, b, k1, k3)):
    '''Simulate m x'' + b x' + k1 x + k3 x^3 = u from rest for a constant set point xd.
    The controller has signature u = controller(x, v, z, xd), where v = x' and z = integral of e.'''
    mt, bt, k1t, k3t = params
    def f(t, s):
        x, v, z = s
        u = controller(x, v, z, xd)
        return [v, (u - bt*v - k1t*x - k3t*x**3) / mt, xd - x]
    t = np.linspace(0, tf, N)
    sol = solve_ivp(f, (0, tf), [0, 0, 0], t_eval=t, rtol=1e-9, atol=1e-11, max_step=0.005)
    u = np.array([controller(*sol.y[:, k], xd) for k in range(N)])
    return t, sol.y[0], sol.y[1], u

def step_metrics(t, x, xd):
    y = x / xd
    overshoot = max(0.0, (y.max() - 1) * 100)
    outside = np.where(np.abs(y - 1) > 0.02)[0]
    ts = t[outside[-1]] if len(outside) else 0.0
    return overshoot, ts

xs = np.linspace(-1.2, 1.2, 241)
plt.figure(figsize=(5, 3))
plt.plot(xs, k1*xs + k3*xs**3, color=C_BLUE, lw=2, label=r'exact: $k_1x+k_3x^3$')
plt.plot(xs, k1*xs, '--', color=C_RED, lw=2, label=r'Taylor linearization at $x=0$: $k_1x$')
plt.xlabel('$x$ (m)'); plt.ylabel('spring force (N)'); plt.legend(); plt.title('Nonlinear spring')
plt.tight_layout(); plt.show()
print('Spring force at x = 1 m: exact %.0f N, linearized %.0f N' % (k1 + k3, k1))

## 1.1 PID control

The PID controller is tuned on the **linear part** $m\ddot x+b\dot x+k_1x$:

$$
u=K_Pe+K_I\int e\,dt+K_D\dot e,\qquad e=x_d-x,
$$

with $K_P=50$, $K_I=100$, $K_D=10$. For a constant set point $\dot e=-\dot x$ (derivative acting on the measurement, which avoids an impulse at $t=0$).

In [ ]:
KP, KI, KD = 50.0, 100.0, 10.0

def pid(x, v, z, xd):
    return KP*(xd - x) + KI*z - KD*v

# Closed-loop poles of the linear part (k3 = 0)
print('Poles with k3 = 0:', np.round(np.roots([1, (b + KD)/m, (k1 + KP)/m, KI/m]), 3))

amps = [0.1, 0.5, 1.0]
cols = [C_GREEN, C_BLUE, C_RED]
plt.figure(figsize=(5.5, 3.2))
for xd, c in zip(amps, cols):
    t, x, _, u = simulate_msd(pid, xd)
    os_, ts = step_metrics(t, x, xd)
    plt.plot(t, x/xd, color=c, lw=2, label=f'$x_d$ = {xd} m (overshoot {os_:.1f}%)')
plt.axhline(1, color='gray', ls=':')
plt.xlim(0, 3); plt.xlabel('$t$ (s)'); plt.ylabel('$x/x_d$'); plt.title('PID: normalised step responses')
plt.legend(); plt.tight_layout(); plt.show()

The same gains produce a different response at each amplitude: the stiffness seen by the controller, $k_1+3k_3x^2$, changes with $x$, so the closed-loop poles move.

## 1.2 Computed-torque (computed-force) control

The force required for a motion $x(t)$ follows from the model (inverse dynamics). Replacing $\ddot x$ by the commanded acceleration $v=\ddot x_d+K_d\dot e+K_pe$:

$$
u=m\big(\ddot x_d+K_d\dot e+K_pe\big)+b\dot x+k_1x+k_3x^3
\quad\Longrightarrow\quad
\ddot e+K_d\dot e+K_pe=0 .
$$

Gains: $K_p=\omega_n^2=25$, $K_d=2\zeta\omega_n=10$ ($\zeta=1$, $\omega_n=5$ rad/s).

## 1.3 Feedback linearization

With $x_1=x$, $x_2=\dot x$: $\ \dot x_1=x_2,\ \ \dot x_2=-\frac1m(bx_2+k_1x_1+k_3x_1^3)+\frac1m u$.

Choosing $u=\alpha(\boldsymbol x)+\beta v$ with $\alpha=bx_2+k_1x_1+k_3x_1^3$, $\beta=m$ gives $\dot x_1=x_2$, $\dot x_2=v$ (exactly linear).
The outer loop $v=\ddot x_d-K_p(x_1-x_d)-K_d(x_2-\dot x_d)$ places the poles at the roots of $s^2+K_ds+K_p$.

The two derivations lead to the same control law; the cell below checks this numerically.

In [ ]:
wn, zeta = 5.0, 1.0
Kp, Kd = wn**2, 2*zeta*wn
# Controller model parameters (equal to the true values unless changed)
m_h, b_h, k1_h, k3_h = m, b, k1, k3

def computed_torque(x, v, z, xd, xd_dot=0.0, xd_ddot=0.0):
    e, e_dot = xd - x, xd_dot - v
    return m_h*(xd_ddot + Kd*e_dot + Kp*e) + b_h*v + k1_h*x + k3_h*x**3

def feedback_linearization(x, v, z, xd, xd_dot=0.0, xd_ddot=0.0):
    x1, x2 = x, v
    alpha = b_h*x2 + k1_h*x1 + k3_h*x1**3
    beta = m_h
    v_cmd = xd_ddot - Kp*(x1 - xd) - Kd*(x2 - xd_dot)
    return alpha + beta*v_cmd

rng = np.random.default_rng(0)
diff = max(abs(computed_torque(*s, 0, xd) - feedback_linearization(*s, 0, xd))
           for s, xd in zip(rng.normal(size=(1000, 2)), rng.normal(size=1000)))
print('Largest difference between the two control laws over 1000 random states: %.1e N' % diff)

## 1.4 Comparison: PID vs. computed torque / feedback linearization

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
rows = []
for xd, c in zip(amps, cols):
    t, x, _, u = simulate_msd(pid, xd)
    ax[0].plot(t, x/xd, color=c, lw=2, label=f'{xd} m')
    os_p, ts_p = step_metrics(t, x, xd); up = np.abs(u).max()
    t, x, _, u = simulate_msd(computed_torque, xd)
    ax[1].plot(t, x/xd, color=c, lw=2, ls=['-', '--', ':'][amps.index(xd)], label=f'{xd} m')
    os_f, ts_f = step_metrics(t, x, xd); uf = np.abs(u).max()
    rows.append((xd, os_p, ts_p, up, os_f, ts_f, uf))
for a, title in zip(ax, ['PID', 'Computed torque / feedback linearization']):
    a.axhline(1, color='gray', ls=':'); a.set_xlim(0, 3); a.set_xlabel('$t$ (s)'); a.set_title(title); a.legend(title='$x_d$')
ax[0].set_ylabel('$x/x_d$'); plt.tight_layout(); plt.show()

print(f"{'x_d (m)':>8} | {'PID overshoot':>13} {'settling':>9} {'peak |u|':>9} | {'CT/FL overshoot':>15} {'settling':>9} {'peak |u|':>9}")
for r in rows:
    print(f"{r[0]:>8} | {r[1]:>12.1f}% {r[2]:>8.2f}s {r[3]:>8.1f}N | {r[4]:>14.1f}% {r[5]:>8.2f}s {r[6]:>8.1f}N")

**Observations**

* PID: overshoot and settling time depend on the amplitude; integral action is required to remove the steady-state error caused by the spring force.
* Computed torque / feedback linearization: the spring and damping forces are cancelled, so the normalised response is identical at every amplitude, with zero steady-state error and no integral action.
* The cancellation requires an accurate model. The interactive cell below lets you introduce a model error in $k_3$ and observe its effect.

### Interactive exploration

* `x_d`: set point.
* `k3_model_ratio`: ratio $\hat k_3/k_3$ used by the computed-torque controller (1 = exact model).
* `wn`: natural frequency of the designed error dynamics.

In [ ]:
def explore_msd(x_d=0.5, k3_model_ratio=1.0, wn_=5.0):
    global m_h, b_h, k1_h, k3_h, Kp, Kd
    Kp, Kd = wn_**2, 2*wn_
    m_h, b_h, k1_h, k3_h = m, b, k1, k3*k3_model_ratio
    tP, xP, _, uP = simulate_msd(pid, x_d)
    tC, xC, _, uC = simulate_msd(computed_torque, x_d)
    fig, ax = plt.subplots(1, 2, figsize=(9, 3))
    ax[0].plot(tP, xP, color=C_RED, lw=2, label='PID')
    ax[0].plot(tC, xC, color=C_BLUE, lw=2, label='computed torque')
    ax[0].axhline(x_d, color='gray', ls=':'); ax[0].set_ylabel('$x$ (m)'); ax[0].set_xlabel('$t$ (s)'); ax[0].legend()
    ax[1].plot(tP, uP, color=C_RED, lw=2); ax[1].plot(tC, uC, color=C_BLUE, lw=2)
    ax[1].set_ylabel('$u$ (N)'); ax[1].set_xlabel('$t$ (s)')
    plt.tight_layout(); plt.show()
    print('Final error: PID %.4f m, computed torque %.4f m' % (x_d - xP[-1], x_d - xC[-1]))
    m_h, b_h, k1_h, k3_h = m, b, k1, k3; Kp, Kd = 25.0, 10.0   # restore defaults

widgets.interact(explore_msd,
                 x_d=widgets.FloatSlider(value=0.5, min=0.05, max=1.5, step=0.05, description='x_d (m)'),
                 k3_model_ratio=widgets.FloatSlider(value=1.0, min=0.0, max=2.0, step=0.1, description='k3_hat/k3'),
                 wn_=widgets.FloatSlider(value=5.0, min=1.0, max=15.0, step=0.5, description='wn (rad/s)'));

---
# Example 2: Two-Link Planar Arm

Point masses $m_1,m_2$ at the link ends; $q_1$ measured from the horizontal, $q_2$ relative to link 1; $m_1=m_2=1$ kg, $l_1=l_2=0.5$ m.

$$
\boldsymbol M(\boldsymbol q)\ddot{\boldsymbol q}+\boldsymbol C(\boldsymbol q,\dot{\boldsymbol q})\dot{\boldsymbol q}+\boldsymbol g(\boldsymbol q)=\boldsymbol\tau
$$

$$
\boldsymbol M=\begin{bmatrix}m_1l_1^2+m_2(l_1^2+2l_1l_2c_2+l_2^2) & m_2(l_1l_2c_2+l_2^2)\\ m_2(l_1l_2c_2+l_2^2) & m_2l_2^2\end{bmatrix},\quad
\boldsymbol C\dot{\boldsymbol q}=\begin{bmatrix}-m_2l_1l_2s_2(2\dot q_1\dot q_2+\dot q_2^2)\\ m_2l_1l_2s_2\dot q_1^2\end{bmatrix},\quad
\boldsymbol g=\begin{bmatrix}(m_1+m_2)gl_1c_1+m_2gl_2c_{12}\\ m_2gl_2c_{12}\end{bmatrix}
$$

In [ ]:
g0 = 9.81
l1 = l2 = 0.5
m1, m2 = 1.0, 1.0

def dynamics(q, qd, m2v=m2):
    '''Return M(q), C(q,qd) qd and g(q) for the two-link arm (end mass m2v).'''
    c2, s2 = np.cos(q[1]), np.sin(q[1])
    M = np.array([[m1*l1**2 + m2v*(l1**2 + 2*l1*l2*c2 + l2**2), m2v*(l1*l2*c2 + l2**2)],
                  [m2v*(l1*l2*c2 + l2**2),                       m2v*l2**2]])
    h = -m2v*l1*l2*s2
    Cqd = np.array([h*(2*qd[0]*qd[1] + qd[1]**2), -h*qd[0]**2])
    G = np.array([(m1 + m2v)*g0*l1*np.cos(q[0]) + m2v*g0*l2*np.cos(q[0] + q[1]),
                  m2v*g0*l2*np.cos(q[0] + q[1])])
    return M, Cqd, G

def quintic(q0, qf, T, t):
    '''Rest-to-rest quintic: position, velocity, acceleration.'''
    s = np.clip(t/T, 0, 1); inside = float(t <= T)
    p = 10*s**3 - 15*s**4 + 6*s**5
    dp = (30*s**2 - 60*s**3 + 30*s**4)/T
    ddp = (60*s - 180*s**2 + 120*s**3)/T**2
    return q0 + (qf - q0)*p, (qf - q0)*dp*inside, (qf - q0)*ddp*inside

def joint_trajectory(qa, qb, T=2.0):
    qa, qb = np.asarray(qa, float), np.asarray(qb, float)
    def ref(t):
        a = quintic(qa[0], qb[0], T, t); c = quintic(qa[1], qb[1], T, t)
        return [np.array([a[i], c[i]]) for i in range(3)]
    return ref

def simulate_arm(controller, ref, x0, tf=3.0, N=601, m2_true=m2):
    '''Integrate the arm under tau = controller(t, q, qd, ref). Returns t, q, qd, e, tau.'''
    def f(t, x):
        q, qd = x[:2], x[2:]
        M, Cqd, G = dynamics(q, qd, m2_true)
        return np.r_[qd, np.linalg.solve(M, controller(t, q, qd, ref) - Cqd - G)]
    t = np.linspace(0, tf, N)
    sol = solve_ivp(f, (0, tf), x0, t_eval=t, rtol=1e-9, atol=1e-11, max_step=0.004)
    q, qd = sol.y[:2], sol.y[2:]
    R = np.array([ref(tt)[0] for tt in t]).T
    tau = np.array([controller(tt, q[:, k], qd[:, k], ref) for k, tt in enumerate(t)]).T
    return t, q, qd, R - q, tau

# Positive definiteness of M over q2
q2s = np.linspace(-np.pi, np.pi, 361)
Ms = np.array([dynamics(np.array([0, a]), np.zeros(2))[0] for a in q2s])
eig_min = min(np.linalg.eigvalsh(Mi).min() for Mi in Ms)
print('Smallest eigenvalue of M over all q2: %.4f kg m^2 (> 0: M positive definite)' % eig_min)
print('M11 ranges from %.2f to %.2f kg m^2' % (Ms[:, 0, 0].min(), Ms[:, 0, 0].max()))

plt.figure(figsize=(5, 3))
plt.plot(q2s/d2r, Ms[:, 0, 0], color=C_BLUE, lw=2, label='$M_{11}$')
plt.plot(q2s/d2r, Ms[:, 0, 1], color=C_ORANGE, lw=2, label='$M_{12}$')
plt.plot(q2s/d2r, Ms[:, 1, 1], '--', color=C_GREEN, lw=2, label='$M_{22}$')
plt.xlabel('$q_2$ (deg)'); plt.ylabel('kg m$^2$'); plt.title('Configuration-dependent inertia'); plt.legend()
plt.tight_layout(); plt.show()

## 2.1 Independent-joint PD control

$\boldsymbol\tau=\boldsymbol K_p\boldsymbol e+\boldsymbol K_d\dot{\boldsymbol e}$ with $\boldsymbol K_p=100\boldsymbol I$ N m/rad, $\boldsymbol K_d=20\boldsymbol I$ N m s/rad.
At rest the PD torque must balance gravity, $\boldsymbol K_p\boldsymbol e_{ss}=\boldsymbol g(\boldsymbol q)$, so the steady-state error depends on the configuration.

The arm starts hanging, $\boldsymbol q(0)=(-90^\circ,0)$, and follows 1.5 s quintic trajectories to three targets.

In [ ]:
def pd_controller(Kp_=100.0, Kd_=20.0):
    def c(t, q, qd, ref):
        r, rd, _ = ref(t)
        return Kp_*(r - q) + Kd_*(rd - qd)
    return c

start = np.array([-90, 0])*d2r
targets = [(-45, 0), (0, 0), (45, 45)]
plt.figure(figsize=(6, 3.2))
print(f"{'target (deg)':>14} | {'e1_ss (deg)':>11} {'e2_ss (deg)':>11}")
for tg, c in zip(targets, [C_GREEN, C_BLUE, C_RED]):
    ref = joint_trajectory(start, np.array(tg)*d2r, 1.5)
    t, q, _, e, _ = simulate_arm(pd_controller(), ref, np.r_[start, 0, 0], tf=2.5)
    R = np.array([ref(tt)[0] for tt in t]).T
    plt.plot(t, R[0]/d2r, ':', color=c); plt.plot(t, q[0]/d2r, color=c, lw=2, label=str(tg))
    print(f"{str(tg):>14} | {e[0, -1]/d2r:>11.1f} {e[1, -1]/d2r:>11.1f}")
plt.xlabel('$t$ (s)'); plt.ylabel('$q_1$ (deg)'); plt.title('PD control (dotted: desired)'); plt.legend(title='target')
plt.tight_layout(); plt.show()

## 2.2 Computed-torque control of the arm

$$
\boldsymbol\tau=\boldsymbol M(\boldsymbol q)\big(\ddot{\boldsymbol q}_d+\boldsymbol K_d\dot{\boldsymbol e}+\boldsymbol K_p\boldsymbol e\big)+\boldsymbol C(\boldsymbol q,\dot{\boldsymbol q})\dot{\boldsymbol q}+\boldsymbol g(\boldsymbol q)
\quad\Longrightarrow\quad
\ddot{\boldsymbol e}+\boldsymbol K_d\dot{\boldsymbol e}+\boldsymbol K_p\boldsymbol e=\boldsymbol 0
$$

The controller may use an estimate $\hat m_2$ of the end mass (to study model errors).

### Gain selection

Each joint error obeys $\ddot e_i+2\zeta\omega_n\dot e_i+\omega_n^2e_i=0$, so $k_{p,i}=\omega_n^2$, $k_{d,i}=2\zeta\omega_n$.

In [ ]:
def ct_controller(Kp_=100.0, Kd_=20.0, m2_model=m2):
    def c(t, q, qd, ref):
        r, rd, rdd = ref(t)
        M, Cqd, G = dynamics(q, qd, m2_model)
        return M @ (rdd + Kd_*(rd - qd) + Kp_*(r - q)) + Cqd + G
    return c

# Designed error response for different damping ratios
tt = np.linspace(0, 0.8, 401); wn = 10.0
plt.figure(figsize=(5, 3))
for z, c in [(0.5, C_RED), (1.0, C_BLUE), (2.0, C_GREEN)]:
    A = np.array([[0, 1], [-wn**2, -2*z*wn]])
    from scipy.linalg import expm
    e = np.array([(expm(A*ti) @ np.array([1.0, 0.0]))[0] for ti in tt])
    plt.plot(tt, e, color=c, lw=2, label=rf'$\zeta$ = {z}  (min {e.min():.2f})')
plt.axhline(0, color='gray', lw=0.8); plt.xlabel('$t$ (s)'); plt.ylabel('$e_i(t)/e_i(0)$')
plt.title(r'Error response, $\omega_n$ = 10 rad/s'); plt.legend(); plt.tight_layout(); plt.show()

## 2.3 Tracking performance: PD vs. computed torque

Quintic trajectories $q_1:0\to60^\circ$, $q_2:0\to90^\circ$ in 2 s, initial error $\boldsymbol e(0)=(10^\circ,-10^\circ)$. Numerical gains 100 and 20 for both controllers.

In [ ]:
ref_task = joint_trajectory([0, 0], np.array([60, 90])*d2r, 2.0)
x0_task = np.array([-10*d2r, 10*d2r, 0, 0])
results = {}
fig, ax = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for a, (name, ctrl) in zip(ax, [('PD', pd_controller()), ('computed torque', ct_controller())]):
    t, q, qd, e, tau = simulate_arm(ctrl, ref_task, x0_task)
    results[name] = (t, q, qd, e, tau)
    a.plot(t, e[0]/d2r, color=C_BLUE, lw=2, label='$e_1$'); a.plot(t, e[1]/d2r, color=C_ORANGE, lw=2, label='$e_2$')
    a.set_title(name); a.set_xlabel('$t$ (s)'); a.legend()
ax[0].set_ylabel('error (deg)'); plt.tight_layout(); plt.show()

for name, (t, q, qd, e, tau) in results.items():
    print(f"{name:>16}: max|e1| for t>=0.5 s = {np.abs(e[0, t >= 0.5]).max()/d2r:6.2f} deg, "
          f"e(3 s) = ({e[0, -1]/d2r:5.2f}, {e[1, -1]/d2r:5.2f}) deg, "
          f"peak |tau| = ({np.abs(tau[0]).max():5.1f}, {np.abs(tau[1]).max():5.1f}) N m")
t, _, _, e, _ = results['computed torque']
print('Designed response e1(0.5 s) = 10*(1+5)*exp(-5) = %.3f deg; simulated %.3f deg' % (10*6*np.exp(-5), e[0, np.argmin(abs(t - 0.5))]/d2r))

## 2.4 Effect of model errors

A payload increases the true end mass to $m_2=1.3$ kg while the controller uses $\hat m_2=1$ kg. The error equation becomes
$\ddot{\boldsymbol e}+\boldsymbol K_d\dot{\boldsymbol e}+\boldsymbol K_p\boldsymbol e=\boldsymbol\eta$; at rest
$\boldsymbol e_{ss}=\boldsymbol K_p^{-1}\hat{\boldsymbol M}^{-1}(\boldsymbol g-\hat{\boldsymbol g})$.

In [ ]:
from scipy.optimize import fsolve
t, q, qd, e, tau = simulate_arm(ct_controller(m2_model=1.0), ref_task, np.zeros(4), tf=4, N=801, m2_true=1.3)

qf = np.array([60, 90])*d2r
def ss_eq(es):
    qq = qf - es
    Mh, _, Gh = dynamics(qq, np.zeros(2), 1.0); _, _, Gt = dynamics(qq, np.zeros(2), 1.3)
    return 100*es - np.linalg.solve(Mh, Gt - Gh)
ess = fsolve(ss_eq, np.zeros(2))

plt.figure(figsize=(5.5, 3))
plt.plot(t, e[0]/d2r, color=C_BLUE, lw=2, label='$e_1$'); plt.plot(t, e[1]/d2r, color=C_ORANGE, lw=2, label='$e_2$')
for v in ess: plt.axhline(v/d2r, color='gray', ls=':')
plt.xlabel('$t$ (s)'); plt.ylabel('error (deg)'); plt.title(r'Computed torque with $\hat m_2$ = 1 kg, true $m_2$ = 1.3 kg')
plt.legend(); plt.tight_layout(); plt.show()
print('Steady-state error: (%.2f, %.2f) deg' % tuple(ess/d2r))

## 2.5 Feedback linearization of the arm

State $\boldsymbol x=[\boldsymbol q;\dot{\boldsymbol q}]$: $\ \dot{\boldsymbol x}=\boldsymbol f(\boldsymbol x)+\boldsymbol G(\boldsymbol x)\boldsymbol\tau$ with
$\boldsymbol f=\begin{bmatrix}\dot{\boldsymbol q}\\-\boldsymbol M^{-1}(\boldsymbol C\dot{\boldsymbol q}+\boldsymbol g)\end{bmatrix}$, $\boldsymbol G=\begin{bmatrix}\boldsymbol 0\\\boldsymbol M^{-1}\end{bmatrix}$.

The feedback $\boldsymbol\tau=\boldsymbol\alpha+\boldsymbol\beta\boldsymbol v$, $\boldsymbol\alpha=\boldsymbol C\dot{\boldsymbol q}+\boldsymbol g$, $\boldsymbol\beta=\boldsymbol M$, gives $\dot{\boldsymbol x}=A\boldsymbol x+B\boldsymbol v$.
The cell verifies numerically that the closed-loop state derivative equals $A\boldsymbol x+B\boldsymbol v$ at random states, and computes the eigenvalues of $A-BK$.

In [ ]:
A = np.block([[np.zeros((2, 2)), np.eye(2)], [np.zeros((2, 2)), np.zeros((2, 2))]])
B = np.vstack([np.zeros((2, 2)), np.eye(2)])

def f_G(x):
    q, qd = x[:2], x[2:]
    M, Cqd, G = dynamics(q, qd)
    Minv = np.linalg.inv(M)
    return np.r_[qd, -Minv @ (Cqd + G)], np.vstack([np.zeros((2, 2)), Minv])

rng = np.random.default_rng(1); worst = 0.0
for _ in range(1000):
    x = rng.uniform(-3, 3, 4); v = rng.uniform(-5, 5, 2)
    M, Cqd, G = dynamics(x[:2], x[2:])
    tau = Cqd + G + M @ v                      # alpha(x) + beta(x) v
    fx, Gx = f_G(x)
    worst = max(worst, np.abs(fx + Gx @ tau - (A @ x + B @ v)).max())
print('max |f(x)+G(x)tau - (Ax+Bv)| over 1000 random states: %.1e' % worst)
print('rank [B AB] =', np.linalg.matrix_rank(np.hstack([B, A @ B])))

for wn in (5, 10, 20):
    K = np.hstack([wn**2*np.eye(2), 2*wn*np.eye(2)])
    print(f'wn = {wn:2d}: eigenvalues of A-BK =', np.round(np.linalg.eigvals(A - B @ K).real, 3))

## 2.6 Outer loop by pole placement

A double pole at $-\omega_n$ for each joint gives $k_{p,i}=\omega_n^2$, $k_{d,i}=2\omega_n$. Faster poles reduce the error faster but require larger torques.

In [ ]:
plt.figure(figsize=(5.5, 3))
print(f"{'wn (rad/s)':>10} | {'e1(0.5 s) (deg)':>15} | {'peak|tau1| (N m)':>16} {'peak|tau2| (N m)':>16}")
for wn, c in [(5, C_RED), (10, C_BLUE), (20, C_GREEN)]:
    t, q, qd, e, tau = simulate_arm(ct_controller(wn**2, 2*wn), ref_task, x0_task)
    plt.plot(t, e[0]/d2r, color=c, lw=2, label=rf'$\omega_n$ = {wn}')
    print(f"{wn:>10} | {e[0, np.argmin(abs(t - 0.5))]/d2r:>15.3f} | {np.abs(tau[0]).max():>16.1f} {np.abs(tau[1]).max():>16.1f}")
plt.xlim(0, 1.5); plt.xlabel('$t$ (s)'); plt.ylabel('$e_1$ (deg)'); plt.title('Effect of pole location'); plt.legend()
plt.tight_layout(); plt.show()

## 2.7 Interactive exploration

* `wn`: natural frequency of the designed error dynamics (computed torque) — PD uses $K_p=\omega_n^2$, $K_d=2\omega_n$ in N m units.
* `payload`: true end mass $m_2$ (the controller always assumes 1 kg).
* `q1_final`, `q2_final`: final configuration of the quintic trajectory.

In [ ]:
def explore_arm(wn=10.0, payload=1.0, q1_final=60.0, q2_final=90.0):
    ref = joint_trajectory([0, 0], np.array([q1_final, q2_final])*d2r, 2.0)
    fig, ax = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
    for a, (name, ctrl) in zip(ax, [('PD', pd_controller(wn**2, 2*wn)), ('computed torque', ct_controller(wn**2, 2*wn, 1.0))]):
        t, q, qd, e, tau = simulate_arm(ctrl, ref, x0_task, m2_true=payload)
        a.plot(t, e[0]/d2r, color=C_BLUE, lw=2, label='$e_1$'); a.plot(t, e[1]/d2r, color=C_ORANGE, lw=2, label='$e_2$')
        a.set_title(f'{name}: e(3 s) = ({e[0, -1]/d2r:.2f}, {e[1, -1]/d2r:.2f}) deg'); a.set_xlabel('$t$ (s)'); a.legend()
    ax[0].set_ylabel('error (deg)'); plt.tight_layout(); plt.show()

widgets.interact(explore_arm,
                 wn=widgets.FloatSlider(value=10, min=2, max=30, step=1, description='wn (rad/s)'),
                 payload=widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.1, description='true m2 (kg)'),
                 q1_final=widgets.FloatSlider(value=60, min=-90, max=90, step=5, description='q1 final'),
                 q2_final=widgets.FloatSlider(value=90, min=-150, max=150, step=5, description='q2 final'));

## 2.8 Animation: PD vs. computed torque

The arm executes the tracking task of Section 2.3 under both controllers. Grey: desired configuration.

In [ ]:
def fk(q):
    p1 = np.array([l1*np.cos(q[0]), l1*np.sin(q[0])])
    p2 = p1 + np.array([l2*np.cos(q[0] + q[1]), l2*np.sin(q[0] + q[1])])
    return np.array([[0, 0], p1, p2])

t, qP = results['PD'][0], results['PD'][1]
qC = results['computed torque'][1]
Rq = np.array([ref_task(tt)[0] for tt in t]).T
idx = np.arange(0, len(t), 6)

fig, axs = plt.subplots(1, 2, figsize=(8, 4))
artists = []
for a, title in zip(axs, ['PD', 'computed torque']):
    a.set_xlim(-1.05, 1.05); a.set_ylim(-1.05, 1.05); a.set_aspect('equal'); a.set_title(title)
    des, = a.plot([], [], 'o-', color='lightgray', lw=4)
    act, = a.plot([], [], 'o-', color=C_BLUE if title != 'PD' else C_RED, lw=3)
    path, = a.plot([], [], color='gray', lw=0.8)
    artists.append((des, act, path))
time_text = fig.suptitle('')

def update(k):
    i = idx[k]
    for (des, act, path), qq in zip(artists, [qP, qC]):
        d = fk(Rq[:, i]); p = fk(qq[:, i])
        des.set_data(d[:, 0], d[:, 1]); act.set_data(p[:, 0], p[:, 1])
        tip = np.array([fk(qq[:, j])[2] for j in range(0, i + 1, 3)])
        path.set_data(tip[:, 0], tip[:, 1])
    time_text.set_text(f't = {t[i]:.2f} s')
    return [x for tr in artists for x in tr]

anim = animation.FuncAnimation(fig, update, frames=len(idx), interval=60, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

---
## Suggested exercises

1. **Example 1**: set $\hat k_3=0$ in the interactive cell (controller ignores the cubic term). Compare the response with PID for $x_d=1$ m.
2. **Example 1**: replace the step set point by a sinusoid $x_d(t)=A\sin\omega t$ (pass $\dot x_d$, $\ddot x_d$ to `computed_torque`) and compare tracking errors of PID and computed torque.
3. **Example 2**: add integral action to the outer loop of the computed-torque controller and repeat Section 2.4. Is the steady-state error removed?
4. **Example 2**: add viscous friction $\boldsymbol F_v\dot{\boldsymbol q}$ to the plant but not to the controller. How large is the resulting tracking error?
5. **Example 2**: execute the same path four times faster ($T=0.5$ s). How do the peak torques change for PD and computed torque?